# Appendix 3.1: Marginal Structural Model

This notebook reviews the additional example of shelving estimating functions with a g-computation estimator of a marginal structural model. The example application here comes from Zivich & Shook-Sa (2025) (also in the `GCompMSM` folder).

In [1]:
import numpy as np
import scipy as sp
import pandas as pd
import delicatessen as deli
from delicatessen import MEstimator
from delicatessen.estimating_equations import ee_regression
from delicatessen.utilities import inverse_logit, aggregate_efuncs
from formulaic import model_matrix

print("Versions")
print("NumPy:       ", np.__version__)
print("SciPy:       ", sp.__version__)
print("Pandas:      ", pd.__version__)
print("Delicatessen:", deli.__version__)

Versions
NumPy:        2.3.5
SciPy:        1.16.3
Pandas:       2.3.3
Delicatessen: 4.3


In [2]:
# Setting up the data
d = pd.read_csv("data/actg.csv")
d['id'] = d.index + 1000

In [3]:
# Model specifications
m_model = ("treat + male + treat:male + idu + white + C(karnof) "
           "+ agec + age_rs1 + age_rs2 + age_rs3 "
           "+ cd4c_0wk + cd4_rs1 + cd4_rs2 + cd4_rs3 "
           "+ male:(cd4c_0wk + cd4_rs1 + cd4_rs2 + cd4_rs3)")
msm_model = "treat + treat:male + male"

In [4]:
# Setup data
d['r'] = 1
d1 = d.copy()                        # Copy original data set
d1['treat'] = 1                      # Set A to 1 in the copy
d1['r'] = 2
d0 = d.copy()                        # Copy original data set
d0['treat'] = 0                      # Set A to 0 in the copy
d0['r'] = 3
ds = pd.concat([d, d1, d0],          # Stacking data sets together
               ignore_index=True)

# Setting up design matrices with stacked data
W = model_matrix(m_model, ds)        # Outcome model design matrix
y = np.asarray(ds['cd4_20wk'])       # Outcome variable
V = model_matrix(msm_model, ds)      # Get MSM design matrix with A=1
r = np.asarray(ds['r'])              # Vector of indicators
idx_msm = V.shape[1]                 # Number of columns in MSM design matrix
idx_m = W.shape[1]                   # Number of columns in outcome model design matrix

In [5]:
def psi_snowden(theta):
    # Defining the estimating function for delicatessen
    beta = theta[:idx_msm]           # Parameters of interest
    alpha = theta[idx_msm:]          # Nuisance parameters

    # Outcome nuisance model
    ee_reg = ee_regression(alpha, X=W, y=y, model='linear') * (r == 1)
    yhat = np.dot(W, alpha)        # Pseudo-outcome under A=1

    # Marginal structural model
    ee_msm = ee_regression(beta, X=V, y=yhat, model='linear') * (r != 1)

    # Returning stacked estimating functions
    return np.vstack([ee_msm, ee_reg])

def psi_cluster(theta):
    # Shelving the estimating function by ID's
    psi_i = psi_snowden(theta=theta)
    return aggregate_efuncs(psi_i, group=ds['id'])

In [6]:
# Applying M-estimator via delicatessen
init_vals = [0., ]*idx_msm + [0., ]*idx_m
estr = MEstimator(psi_cluster, init=init_vals)
estr.estimate()
estr.print_results(subset=[0, 1, 2, 3], decimals=2)

              Estimation Method: M-estimator
--------------------------------------------------------------
No. Observations:        1578 | No. Parameters:             24
Solving algorithm:         lm | Max Iterations:           5000
Solving tolerance:      1e-09 | Allow P-Inverse:             1
Derivative Method:     approx | Deriv Approx:            1e-09
Small N Correction:      None | Distribution:           Z-stat
   Theta   StdErr  Z-score      LCL      UCL  P-value  S-value 
--------------------------------------------------------------
  347.20    12.21    28.43   323.26   371.14     0.00   588.10 
   57.13    14.19     4.03    29.31    84.94     0.00    14.10 
  -15.13    13.32    -1.14   -41.23    10.97     0.26     1.97 
   -4.28    15.50    -0.28   -34.66    26.10     0.78     0.35 
